# Predicting Departure Delays: Lasso, Ridge, GBT, and Shared Evaluation

This notebook focuses on the portion of the modeling pipeline assigned to Banjot Saini: regularized linear regression through Lasso and Ridge, Gradient Boosted Trees, and end-to-end evaluation on a shared hold-out framework.

Consistent with the methodology outlined in the Phase 1 deliverable, the task is framed as a regression problem with `DEP_DELAY` as the target variable. The joined OTPW-style dataset is used to construct features available prior to scheduled departure, apply a chronological train/validation/test split, tune model hyperparameters on the validation window, and compare final model performance on a held-out test set.

The shared metrics reported in this notebook are `RMSE`, `MAE`, `R2`, `OTPA`, `SDDR`, and `F1`. The data path and execution flow are also aligned with the Databricks EDA notebooks so that this notebook can be run in the same workspace environment.

In [ ]:
import sys
from pathlib import Path

candidate_dirs = [Path.cwd(), Path.cwd() / "notebook_code", Path.cwd().parent / "notebook_code"]
for candidate in candidate_dirs:
    if (candidate / "modeling_utils.py").exists():
        candidate_str = str(candidate)
        if candidate_str not in sys.path:
            sys.path.append(candidate_str)
        break
else:
    raise FileNotFoundError("Could not find notebook_code/modeling_utils.py from the current working directory.")

import pandas as pd

IS_DATABRICKS = "dbutils" in globals()
print(f"Running in Databricks: {IS_DATABRICKS}")

try:
    from modeling_utils import (
        DEFAULT_DELAY_THRESHOLD,
        DEFAULT_SEVERE_DELAY_THRESHOLD,
        describe_splits,
        evaluate_model,
        fit_gbt_model,
        fit_linear_model,
        infer_time_split_boundaries,
        prepare_modeling_frame,
        prune_empty_feature_columns,
        resolve_feature_columns,
        search_gbt_model,
        search_linear_model,
        time_based_split,
        top_gbt_importances,
        top_linear_coefficients,
    )
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Could not import modeling_utils.py. If you are running this in Databricks, keep the notebook and modeling_utils.py in the same repo/workspace folder structure before running it."
    ) from exc


## Configuration

This section defines the primary notebook parameters, including the Databricks data path, temporal split boundaries, sampling options, and model-specific hyperparameter grids. If the team updates the underlying dataset or revises the split strategy, this is the main cell to modify.

In [ ]:
data_BASE_DIR = "dbfs:/mnt/mids-w261/"
# If you want to sanity-check the mount in Databricks, uncomment the next line.
# display(dbutils.fs.ls(data_BASE_DIR))

DATA_PATH = f"{data_BASE_DIR}/OTPW_60M/OTPW_60M/"
ROW_FILTER = "YEAR = 2019 AND QUARTER = 4"  # If the upstream table is already filtered, set this to None.
DATE_COL = "FL_DATE"
TARGET_COL = "DEP_DELAY"

TRAIN_END = "2019-11-15"
VALID_END = "2019-11-30"
AUTO_INFER_SPLITS = False  # Turn this on if you want the notebook to infer split cutoffs from the current data.

SAMPLE_FRACTION = None  # Helpful for quick iteration on-cluster. Example: 0.05.
SEED = 42

DELAY_THRESHOLD = DEFAULT_DELAY_THRESHOLD
SEVERE_DELAY_THRESHOLD = DEFAULT_SEVERE_DELAY_THRESHOLD

LASSO_REG_PARAMS = [0.001, 0.01, 0.05]
RIDGE_REG_PARAMS = [0.01, 0.10, 1.00]
LINEAR_MAX_ITER = 100

GBT_PARAM_GRID = [
    {"maxDepth": 5, "maxIter": 40, "stepSize": 0.05},
    {"maxDepth": 7, "maxIter": 60, "stepSize": 0.05},
    {"maxDepth": 5, "maxIter": 80, "stepSize": 0.10},
]


## Data Ingestion And Feature Preparation

The joined OTPW dataset is loaded from DBFS and filtered to the selected analysis window. The preparation step then retains the prediction target, removes records that would introduce invalid supervision, and transforms flight and weather fields into model-ready numeric and categorical inputs.

In [ ]:
raw_df = spark.read.parquet(DATA_PATH)
if ROW_FILTER:
    raw_df = raw_df.filter(ROW_FILTER)

model_df = prepare_modeling_frame(
    raw_df,
    target_col=TARGET_COL,
    date_col=DATE_COL,
    sample_fraction=SAMPLE_FRACTION,
    seed=SEED,
)

numeric_cols, linear_cat_cols, gbt_cat_cols = resolve_feature_columns(model_df)
print("Numeric features ready for modeling:", numeric_cols)
print("Categorical features for the linear models:", linear_cat_cols)
print("Categorical features for GBT:", gbt_cat_cols)

preview_df = model_df.limit(5)
if "display" in globals():
    display(preview_df)
else:
    preview_df.show(truncate=False)


## Train/Test/Validation Split

Following the Phase 1 methodology, the dataset is partitioned chronologically to prevent leakage from future observations into past predictions. This split structure preserves temporal ordering, supports validation-based hyperparameter tuning, and ensures that final model performance is measured on a held-out test period. An additional guard clause is included so the notebook fails fast if any split is empty.

In [ ]:
if AUTO_INFER_SPLITS or TRAIN_END is None or VALID_END is None:
    TRAIN_END, VALID_END = infer_time_split_boundaries(model_df, date_col=DATE_COL)

train_df, valid_df, test_df = time_based_split(
    model_df,
    date_col=DATE_COL,
    train_end=TRAIN_END,
    valid_end=VALID_END,
)

train_df = train_df.cache()
valid_df = valid_df.cache()
test_df = test_df.cache()

split_summary = describe_splits(train_df, valid_df, test_df, date_col=DATE_COL)
if (split_summary["rows"] == 0).any():
    raise ValueError("At least one split is empty. Adjust ROW_FILTER or the split boundaries before training.")

linear_numeric_cols, linear_cat_cols = prune_empty_feature_columns(train_df, numeric_cols, linear_cat_cols)
gbt_numeric_cols, gbt_cat_cols = prune_empty_feature_columns(train_df, numeric_cols, gbt_cat_cols)

print("Split summary")
print(split_summary.to_string(index=False))
print("Usable numeric features for the linear models:", linear_numeric_cols)
print("Usable categorical features for the linear models:", linear_cat_cols)
print("Usable numeric features for GBT:", gbt_numeric_cols)
print("Usable categorical features for GBT:", gbt_cat_cols)

split_summary


## Model Training And Validation

This stage trains the three assigned models on the training split and evaluates candidate hyperparameter settings on the validation split. Lasso and Ridge provide regularized linear baselines for high-dimensional and potentially collinear feature spaces, while Gradient Boosted Trees are included to capture the non-linear and threshold-based relationships identified in the earlier EDA. The resulting validation summary table supports direct model comparison before final retraining.

In [ ]:
best_lasso, lasso_search = search_linear_model(
    train_df=train_df,
    valid_df=valid_df,
    numeric_columns=linear_numeric_cols,
    categorical_columns=linear_cat_cols,
    reg_params=LASSO_REG_PARAMS,
    elastic_net_param=1.0,
    model_name="Lasso",
    max_iter=LINEAR_MAX_ITER,
    delay_threshold=DELAY_THRESHOLD,
    severe_delay_threshold=SEVERE_DELAY_THRESHOLD,
)

best_ridge, ridge_search = search_linear_model(
    train_df=train_df,
    valid_df=valid_df,
    numeric_columns=linear_numeric_cols,
    categorical_columns=linear_cat_cols,
    reg_params=RIDGE_REG_PARAMS,
    elastic_net_param=0.0,
    model_name="Ridge",
    max_iter=LINEAR_MAX_ITER,
    delay_threshold=DELAY_THRESHOLD,
    severe_delay_threshold=SEVERE_DELAY_THRESHOLD,
)

best_gbt, gbt_search = search_gbt_model(
    train_df=train_df,
    valid_df=valid_df,
    numeric_columns=gbt_numeric_cols,
    categorical_columns=gbt_cat_cols,
    param_grid=GBT_PARAM_GRID,
    model_name="GBT",
    delay_threshold=DELAY_THRESHOLD,
    severe_delay_threshold=SEVERE_DELAY_THRESHOLD,
    seed=SEED,
)

validation_search = (
    pd.concat([lasso_search, ridge_search, gbt_search], ignore_index=True, sort=False)
    .sort_values(["RMSE", "model"])
    .reset_index(drop=True)
)

validation_search


## Final Model Evaluation

After selecting the best validation configuration for each model, the training and validation windows are combined and each model is refit on the enlarged development set. Final performance is then measured on the held-out test set using the shared evaluation metrics defined in the project methodology.

In [ ]:
train_valid_df = train_df.unionByName(valid_df).cache()

lasso_model = fit_linear_model(
    train_df=train_valid_df,
    numeric_columns=linear_numeric_cols,
    categorical_columns=linear_cat_cols,
    reg_param=best_lasso["regParam"],
    elastic_net_param=best_lasso["elasticNetParam"],
    max_iter=int(best_lasso["maxIter"]),
)

ridge_model = fit_linear_model(
    train_df=train_valid_df,
    numeric_columns=linear_numeric_cols,
    categorical_columns=linear_cat_cols,
    reg_param=best_ridge["regParam"],
    elastic_net_param=best_ridge["elasticNetParam"],
    max_iter=int(best_ridge["maxIter"]),
)

gbt_model = fit_gbt_model(
    train_df=train_valid_df,
    numeric_columns=gbt_numeric_cols,
    categorical_columns=gbt_cat_cols,
    max_depth=int(best_gbt["maxDepth"]),
    max_iter=int(best_gbt["maxIter"]),
    step_size=float(best_gbt["stepSize"]),
    seed=SEED,
)

test_results = pd.DataFrame(
    [
        {
            **evaluate_model(lasso_model, test_df, model_name="Lasso", split_name="test", delay_threshold=DELAY_THRESHOLD, severe_delay_threshold=SEVERE_DELAY_THRESHOLD),
            "regParam": best_lasso["regParam"],
            "elasticNetParam": best_lasso["elasticNetParam"],
            "maxIter": best_lasso["maxIter"],
        },
        {
            **evaluate_model(ridge_model, test_df, model_name="Ridge", split_name="test", delay_threshold=DELAY_THRESHOLD, severe_delay_threshold=SEVERE_DELAY_THRESHOLD),
            "regParam": best_ridge["regParam"],
            "elasticNetParam": best_ridge["elasticNetParam"],
            "maxIter": best_ridge["maxIter"],
        },
        {
            **evaluate_model(gbt_model, test_df, model_name="GBT", split_name="test", delay_threshold=DELAY_THRESHOLD, severe_delay_threshold=SEVERE_DELAY_THRESHOLD),
            "maxDepth": best_gbt["maxDepth"],
            "maxIter": best_gbt["maxIter"],
            "stepSize": best_gbt["stepSize"],
        },
    ]
).sort_values(["RMSE", "model"]).reset_index(drop=True)

test_results


## Model Interpretation

To support presentation and downstream discussion, the notebook also surfaces the strongest learned signals for each fitted model. For the linear models, this is done through the largest coefficients in absolute value; for Gradient Boosted Trees, this is done through ranked feature importances.

In [ ]:
lasso_features = top_linear_coefficients(lasso_model, train_valid_df, top_n=20)
ridge_features = top_linear_coefficients(ridge_model, train_valid_df, top_n=20)
gbt_features = top_gbt_importances(gbt_model, train_valid_df, top_n=20)

for name, table in [("Lasso", lasso_features), ("Ridge", ridge_features), ("GBT", gbt_features)]:
    print(f"\nTop signals for {name}\n")
    if "display" in globals():
        display(table)
    else:
        print(table.to_string(index=False))
